Import Canvas API, URL and Key, and test connection

In [3]:
from canvasapi import Canvas

#Replace these with your information
API_URL = "https://psu.instructure.com"
API_KEY = "1050~eY64ZGTthKW7tkVEKL9nwanKwQavTkV6ehBrtCUTRcnXvLR3GLGaFuEDmQ62TBG9"

canvas = Canvas(API_URL, API_KEY)

#test connection
user = canvas.get_current_user()
print(f"Connected as {user.name}")

Connected as Heather Zehnder


Fetch courses. You will need the ID number(s) for later steps.

In [4]:
courses = canvas.get_courses(enrollment_state="active")
print("Active Courses:")
for course in courses:
    name = getattr(course, "name", "Unnamed Course")
    print(f"ID: {course.id} - Name: {name}")

Active Courses:
ID: 2383222 - Name: [25SP] NUTR 100, Section 001: NUTR APP HEALTH (WC, Borkowska)
ID: 2421682 - Name: 25FA CAS 100A, MRG: Effective Speech (WC, Serber)
ID: 2448712 - Name: Active Attacker Response
ID: 2382739 - Name: DA 101: INTRO DA - SP 25 (ER & WC Merged)
ID: 2419584 - Name: DA 305: Data Ethics and Privacy FA 25 (WC & ER Merged)
ID: 2450267 - Name: Descriptive Analytics
ID: 2450274 - Name: Diagnostic Analytics
ID: 2448830 - Name: Intro to R
ID: 2452075 - Name: Organizing Data
ID: 218420000000000333 - Name: Prepare for Success: Penn State World Campus Pre-Orientation (SP25)
ID: 2417497 - Name: PSYCH 200, Section 001, FA25: Elem Stat Psych


Target particular course

In [ ]:
#replace INSERT_COURSE_ID_HERE with the course ID(s) you want to access. Separate multiple IDs with commas.
COURSE_IDS = [INSERT_COURSE_ID_HERE]
for COURSE_ID in COURSE_IDS:
    course = canvas.get_course(COURSE_ID)
    print(f"Course Name: {course.name}")

Course Name: [25SP] NUTR 100, Section 001: NUTR APP HEALTH (WC, Borkowska)
Course Name: 25FA CAS 100A, MRG: Effective Speech (WC, Serber)


Identify items in module - this step is optional

In [ ]:
for COURSE_ID in COURSE_IDS:
    course = canvas.get_course(COURSE_ID)
    
    modules = course.get_modules()
    print(f"\n\nModiules for Course: {course.name}")
    for module in modules:
        print(f"Module: {module.name}")
        items = module.get_module_items()
        for item in items:
            content_type = item.type
            title = item.title
            
            if content_type == "ExternalUrl":
                print(f"[LINK] {title}: {item.external_url}")
            elif content_type == "File":
                print(f"[FILE] {title} (ID: {item.content_id})")
            elif content_type == "Page":
                print(f"[PAGE] {title}")
            else:
                print(f"[{content_type.upper()}] {title}")   
            

Modiules for Course: [25SP] NUTR 100, Section 001: NUTR APP HEALTH (WC, Borkowska)
Module: Course Orientation
[PAGE] Welcome to NUTR 100! Overview and Instructions
[LINK] Student Health, Well-Being, Career, and Financial Resources: https://docs.google.com/document/d/1eyVUWP-LOBHGJBgeKs5wYAvCyL-lzNgpAgVx-CW9lVM/edit?usp=sharing
[PAGE] Course Structure and Expectations
[PAGE] Lesson Readings and Library Resources
[PAGE] Writing Resources
[PAGE] Course Communication With Instructor and Classmates
[PAGE] Syllabus Review and Due Dates
[PAGE] Getting Started with Canvas
[PAGE] Tips for Success
[PAGE] Course Assignments Overview
[PAGE] Quizzes
[PAGE] Personal Nutrition Portfolio Overview
[SUBHEADER] PackBack Discussion Assignments
[PAGE] PackBack Discussion Assignments
[PAGE] PackBack Posting Requirements
[PAGE] PackBack Posting Guidelines
[PAGE] PackBack Grading Requirements
[SUBHEADER] COURSE ORIENTATION ASSIGNMENTS
[ASSIGNMENT] Introduction PackBack Discussion
[QUIZ] Syllabus and Orientati

Download items and save any external links

In [8]:
import os
import re
from canvasapi import Canvas


# DEFAULT SAVE FILEPATH IS DESKTOP
desktop = os.path.join(os.path.expanduser("~"), "Desktop")

for COURSE_ID in COURSE_IDS:
    
    output_dir = os.path.join(desktop, f"Canvas_Backup_{COURSE_ID}")

        #if you want to specify the directory, uncomment and edit one of the lines below
        #windows 
        #output_dir = r"C:\Users\YourName\Desktop\Canvas_Backup"
        #mac/linux
        #output_dir = "/home/YourName/Desktop/Canvas_Backup"

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # re-initialize if needed 
    canvas = Canvas(API_URL, API_KEY)
    course = canvas.get_course(COURSE_ID)
    modules = course.get_modules()

    def sanitize_name(name):
        """cleans filenames to play nice with the OS"""
        clean = re.sub(r'[<>:"/\\|?*]', '_', name).strip()
        return clean.rstrip('.')

    # main scraping loop
    print(f"Starting scrape for: {course.name}...\n")

    for module in modules:
        clean_mod_name = sanitize_name(module.name)
        module_path = os.path.join(output_dir, clean_mod_name)
        
        if not os.path.exists(module_path):
            os.makedirs(module_path)
        
        print(f"Processing Module: {module.name}")
        items = module.get_module_items()
        
        for item in items:
            clean_title = sanitize_name(item.title)
            
            # FILES (PDF, PPT, MP4, etc.)
            if item.type == "File":
                try:
                    canvas_file = canvas.get_file(item.content_id)
                    file_dest = os.path.join(module_path, canvas_file.filename)
                    canvas_file.download(file_dest)
                    print(f"   [FILE] Downloaded: {canvas_file.filename}")
                except Exception as e:
                    print(f"   [ERROR] Could not download file {item.title}: {e}")
            
            # PAGES (HTML Content) ---
            elif item.type == "Page":
                try:
                    page = course.get_page(item.page_url)
                    page_path = os.path.join(module_path, f"{clean_title}.html")
                    with open(page_path, "w", encoding="utf-8") as f:
                        f.write(f"<html><head><title>{item.title}</title></head><body>")
                        f.write(page.body if page.body else "<em>No content</em>")
                        f.write("</body></html>")
                    print(f"   [PAGE] Scraped HTML: {clean_title}")
                except Exception as e:
                    print(f"   [ERROR] Could not scrape page {item.title}: {e}")

            # EXTERNAL LINKS 
            elif item.type == "ExternalUrl":
                links_file = os.path.join(module_path, "external_links.txt")
                with open(links_file, "a", encoding="utf-8") as f:
                    f.write(f"{item.title}: {item.external_url}\n")
                print(f"   [LINK] Saved: {item.title}")

            # EXTERNAL TOOLS (LTI)
            elif item.type == "ExternalTool":
                links_file = os.path.join(module_path, "external_tools_to_check.txt")
                with open(links_file, "a", encoding="utf-8") as f:
                    f.write(f"{item.title}: {item.html_url}\n")
                print(f"   [TOOL] Noted Tool: {item.title}")

print(f"\nDone! Check your save folder: {output_dir}.")

Starting scrape for: [25SP] NUTR 100, Section 001: NUTR APP HEALTH (WC, Borkowska)...

Processing Module: Course Orientation
   [PAGE] Scraped HTML: Welcome to NUTR 100! Overview and Instructions
   [LINK] Saved: Student Health, Well-Being, Career, and Financial Resources
   [PAGE] Scraped HTML: Course Structure and Expectations
   [PAGE] Scraped HTML: Lesson Readings and Library Resources
   [PAGE] Scraped HTML: Writing Resources
   [PAGE] Scraped HTML: Course Communication With Instructor and Classmates
   [PAGE] Scraped HTML: Syllabus Review and Due Dates
   [PAGE] Scraped HTML: Getting Started with Canvas
   [PAGE] Scraped HTML: Tips for Success
   [PAGE] Scraped HTML: Course Assignments Overview
   [PAGE] Scraped HTML: Quizzes
   [PAGE] Scraped HTML: Personal Nutrition Portfolio Overview
   [PAGE] Scraped HTML: PackBack Discussion Assignments
   [PAGE] Scraped HTML: PackBack Posting Requirements
   [PAGE] Scraped HTML: PackBack Posting Guidelines
   [PAGE] Scraped HTML: PackBack G